In [2]:
# Robust CLV target generation (handles different datetime / amount column names)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

# ---------------------------
# Paths (edit if needed)
# ---------------------------
CLV_DF_PATH = Path("C:\\Users\\abhin\\Desktop\\Projects\\FinTech Projects\\Prophet\\Dataset\\clv_7d_ready_dataset.csv")
EVENTS_PATH   = Path("C:\\Users\\abhin\\Desktop\\Projects\\FinTech Projects\\Prophet\\Dataset\\cleaned_transcript_synthetic.csv")
OUT_PATH      = Path("C:\\Users\\abhin\\Desktop\\Projects\\FinTech Projects\\Prophet\\Dataset\\clv_dataset_with_true_spend.csv")

print("CLV_DF_PATH:", CLV_DF_PATH)
print("EVENTS_PATH:", EVENTS_PATH)

# ---------------------------
# Load anchor dataset
# ---------------------------
print("Loading CLV base dataset...")
df = pd.read_csv(CLV_DF_PATH, parse_dates=["anchor_time"], dayfirst=False)
print("CLV dataset shape:", df.shape)

# ---------------------------
# Load events (no parse_dates — we'll detect and convert)
# ---------------------------
print("Loading event transcript (raw)...")
events = pd.read_csv(EVENTS_PATH, low_memory=False)
print("Events raw shape:", events.shape)
print("Events columns:", list(events.columns)[:20])

# ---------------------------
# Detect datetime column in events
# ---------------------------
possible_time_cols = ["ts", "time", "event_time", "time_parsed", "event_time_parsed", "timestamp", "datetime"]
time_col = None
for c in possible_time_cols:
    if c in events.columns:
        time_col = c
        break
# If nothing found, try to heuristically find any column with 'time'/'date' in name
if time_col is None:
    for c in events.columns:
        if ("time" in c.lower() or "date" in c.lower() or "ts" == c.lower()):
            time_col = c
            break

if time_col is None:
    raise ValueError("Could not find a datetime-like column in events CSV. Columns: %s" % list(events.columns)[:40])

print("Detected event time column:", time_col)

# Parse it robustly (mixed formats supported)
events[time_col] = pd.to_datetime(events[time_col], errors="coerce", infer_datetime_format=True)
n_missing_dt = events[time_col].isna().sum()
if n_missing_dt:
    print(f"Warning: {n_missing_dt} rows in events have unparsable datetimes (set to NaT).")

# Normalize name to 'event_time'
events = events.rename(columns={time_col: "event_time"})

# ---------------------------
# Ensure person column exists
# ---------------------------
if "person" not in events.columns:
    # try alternatives
    for alt in ["customer", "user", "customer_id", "person_id"]:
        if alt in events.columns:
            events = events.rename(columns={alt: "person"})
            break
if "person" not in events.columns:
    raise ValueError("No 'person' column found in events file. Found columns: %s" % list(events.columns)[:40])

# ---------------------------
# Detect transaction rows and amount column
# ---------------------------
# Common event column names: 'event', 'event_type', 'type'
evt_col = None
for c in ["event", "event_type", "type", "action"]:
    if c in events.columns:
        evt_col = c
        break
if evt_col is None:
    raise ValueError("No event-type column found. Expected one of ['event','event_type','type','action'].")

# Detect amount/spend column
amt_col = None
for c in events.columns:
    if "amount" in c.lower() or "spend" in c.lower() or "price" in c.lower() or "value" in c.lower():
        amt_col = c
        break
# If no numeric amount column found, create amount=0 for non-transactions (and try to use other columns)
if amt_col is None:
    print("No explicit amount column found. Creating 'amount' and filling 1.0 for 'transaction' rows if you want counts instead.")
    events["amount"] = 0.0
    amt_col = "amount"

# Keep only rows where event_time not null
events = events[~events["event_time"].isna()].copy()

# ---------------------------
# Filter transaction-like events
# ---------------------------
tx_mask = events[evt_col].astype(str).str.lower().str.contains("transaction|purchase|order|paid", na=False)
tx = events[tx_mask].copy()
if tx.empty:
    print("Warning: no transaction-like rows found; will treat 'amount' where >0 as transactions.")
    tx = events[events[amt_col].notna() & (events[amt_col] != 0)].copy()

# Ensure amount numeric
tx[amt_col] = pd.to_numeric(tx[amt_col], errors="coerce").fillna(0.0)
tx = tx.sort_values(["person", "event_time"]).reset_index(drop=True)
print("Transaction rows after filter:", tx.shape)

# rename amount col to 'amount' for consistency
if amt_col != "amount":
    tx = tx.rename(columns={amt_col: "amount"})
else:
    tx = tx.copy()

# ---------------------------
# Group transactions by person for fast lookup
# ---------------------------
print("Building person -> events groups (for fast lookup)...")
groups = {pid: grp[["event_time", "amount"]].reset_index(drop=True) for pid, grp in tx.groupby("person")}
print("Unique persons with transactions:", len(groups))

# ---------------------------
# Prepare output columns on anchor df
# ---------------------------
df["future_30d_spend_true"] = 0.0
df["future_90d_spend_true"] = 0.0
df["purchase_next_30d_true"] = 0
df["purchase_next_90d_true"] = 0

# ---------------------------
# Compute function (vectorized-ish)
# ---------------------------
def compute_future_for_row(person, anchor_time):
    if person not in groups:
        return 0.0, 0.0, 0, 0
    g = groups[person]
    # g is a small DataFrame for that person; use boolean mask
    start = anchor_time
    end30 = anchor_time + pd.Timedelta(days=30)
    end90 = anchor_time + pd.Timedelta(days=90)

    # boolean masks
    m30 = (g["event_time"] >= start) & (g["event_time"] < end30)
    m90 = (g["event_time"] >= start) & (g["event_time"] < end90)

    spend30 = float(g.loc[m30, "amount"].sum())
    spend90 = float(g.loc[m90, "amount"].sum())
    purch30 = int(spend30 > 0.0)
    purch90 = int(spend90 > 0.0)
    return spend30, spend90, purch30, purch90

# ---------------------------
# Iterate anchors with progress bar (fast)
# ---------------------------
print("Computing true future spend & purchase flags for each anchor... (this may take a while)")
tqdm.pandas()
res = df.progress_apply(lambda r: compute_future_for_row(r["person"], r["anchor_time"]), axis=1)

# res is series of tuples — convert to DataFrame
res_df = pd.DataFrame(res.tolist(), columns=["future_30d_spend_true","future_90d_spend_true","purchase_next_30d_true","purchase_next_90d_true"])
df[res_df.columns] = res_df

# ---------------------------
# Save output
# ---------------------------
df.to_csv(OUT_PATH, index=False)
print(f"Saved CLV dataset with true spend -> {OUT_PATH}")
print("\nTargets summary (counts & stats):")
print(df[["future_30d_spend_true","future_90d_spend_true","purchase_next_30d_true","purchase_next_90d_true"]].describe(include="all"))



CLV_DF_PATH: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_7d_ready_dataset.csv
EVENTS_PATH: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\cleaned_transcript_synthetic.csv
Loading CLV base dataset...
CLV dataset shape: (65911, 135)
Loading event transcript (raw)...
Events raw shape: (6567251, 8)
Events columns: ['person', 'time', 'event', 'amount', 'product_id', 'offer_id', 'ad_exposed', 'time_parsed']
Detected event time column: time


C:\Users\abhin\AppData\Local\Temp\ipykernel_10628\1564920517.py:54: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  events[time_col] = pd.to_datetime(events[time_col], errors="coerce", infer_datetime_format=True)


Transaction rows after filter: (192983, 8)
Building person -> events groups (for fast lookup)...
Unique persons with transactions: 29800
Computing true future spend & purchase flags for each anchor... (this may take a while)


100%|██████████| 65911/65911 [01:11<00:00, 928.12it/s] 


Saved CLV dataset with true spend -> C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv

Targets summary (counts & stats):
       future_30d_spend_true  future_90d_spend_true  purchase_next_30d_true  \
count           65911.000000           65911.000000            65911.000000   
mean               35.489677              66.166107                0.300041   
std                66.855158              99.318890                0.458279   
min                 0.000000               0.000000                0.000000   
25%                 0.000000               0.000000                0.000000   
50%                 0.000000               0.000000                0.000000   
75%                64.316923             101.315162                1.000000   
max               910.962569            1583.579754                1.000000   

       purchase_next_90d_true  
count            65911.000000  
mean                 0.466447  
std                  0.

In [7]:
"""
CLV_MODEL_FINAL.py  
Train a single-stage LightGBM CLV model using your consolidated CLV dataset:
    clv_dataset_with_true_spend.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
CLV_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv"
OUT_DIR = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\final_clv_model")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "future_30d_spend_true"   # change to 30d, 90d, 7d, etc. as needed
TEST_SIZE = 0.2

# LightGBM parameters
LGB_TWEEDIE_PARAMS = dict(
    objective="tweedie",
    tweedie_variance_power=1.4,
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

LGB_LOG_PARAMS = dict(
    objective="regression",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

EARLY_STOPPING = 100
VERBOSE = 100

# ============================================================
# Helper functions
# ============================================================
def downcast(df):
    for c in df.select_dtypes(include=['int64']).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    for c in df.select_dtypes(include=['float64']).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    return df

def top_decile_lift(y_true, y_score, decile=0.1):
    n = len(y_true)
    k = int(n * decile)
    idx = np.argsort(-y_score)[:k]
    top_sum = y_true[idx].sum()
    baseline = y_true.mean() * k
    lift = top_sum / baseline
    return lift, top_sum, baseline

# ============================================================
# 1) Load dataset
# ============================================================
print(f"Loading CLV dataset: {CLV_PATH}")
df = pd.read_csv(CLV_PATH)
print("Dataset shape:", df.shape)

if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' not found! Available columns:\n{df.columns.tolist()}")

y = df[TARGET].astype(float).values

# Remove ID/time/label columns
drop_cols = [TARGET, "person", "anchor_time", "time", "time_parsed"]
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# Encode categoricals
for col in X_df.select_dtypes(exclude=[np.number]).columns:
    X_df[col] = pd.factorize(X_df[col].astype(str))[0]

X_df = downcast(X_df)

# Imputation
imp = SimpleImputer(strategy="median")
X = pd.DataFrame(imp.fit_transform(X_df), columns=X_df.columns).astype("float32")
joblib.dump(imp, OUT_DIR / "imputer.joblib")

# ============================================================
# 2) Train / test split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# ============================================================
# 3) Model A: Tweedie Regressor
# ============================================================
print("\nTraining Tweedie model...")
model_tweedie = lgb.LGBMRegressor(**LGB_TWEEDIE_PARAMS)
model_tweedie.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="rmse",
)

pred_tweedie = np.clip(model_tweedie.predict(X_test), 0, None)

rmse_t = mean_squared_error(y_test, pred_tweedie)**0.5
mae_t = mean_absolute_error(y_test, pred_tweedie)
r2_t = r2_score(y_test, pred_tweedie)
lift_t, ts_t, bl_t = top_decile_lift(y_test, pred_tweedie)

print(f"Tweedie RMSE={rmse_t:.4f} MAE={mae_t:.4f} R2={r2_t:.4f} Lift={lift_t:.3f}")

# ============================================================
# 4) Model B: Log1p Regressor
# ============================================================
print("\nTraining log1p model...")
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

model_log = lgb.LGBMRegressor(**LGB_LOG_PARAMS)
model_log.fit(
    X_train, y_train_log,
    eval_set=[(X_test, y_test_log)],
    eval_metric="rmse",
)

pred_log = np.clip(np.expm1(model_log.predict(X_test)), 0, None)

rmse_l = mean_squared_error(y_test, pred_log)**0.5
mae_l = mean_absolute_error(y_test, pred_log)
r2_l = r2_score(y_test, pred_log)
lift_l, ts_l, bl_l = top_decile_lift(y_test, pred_log)

print(f"Log1p RMSE={rmse_l:.4f} MAE={mae_l:.4f} R2={r2_l:.4f} Lift={lift_l:.3f}")

# ============================================================
# 5) Select best model
# ============================================================
if rmse_t <= rmse_l:
    best_model = model_tweedie
    best_name = "tweedie"
    best_preds = pred_tweedie
    best_metrics = (rmse_t, mae_t, r2_t, lift_t)
else:
    best_model = model_log
    best_name = "log1p"
    best_preds = pred_log
    best_metrics = (rmse_l, mae_l, r2_l, lift_l)

print("\n============================")
print(f"Selected Best Model: {best_name.upper()}")
print(f"RMSE={best_metrics[0]:.4f} MAE={best_metrics[1]:.4f} R2={best_metrics[2]:.4f} LIFT={best_metrics[3]:.3f}")
print("============================")

# Save model
joblib.dump(best_model, OUT_DIR / f"clv_model_{best_name}.joblib")

# ============================================================
# 6) Save full predictions
# ============================================================
full_preds = best_model.predict(X)
if best_name == "log1p":
    full_preds = np.expm1(full_preds)
full_preds = np.clip(full_preds, 0, None)

out = df[["person"]].copy()
out["predicted_clv_30d"] = full_preds
out["actual_30d"] = y

out.to_csv(OUT_DIR / "CLV_predictions_full.csv", index=False)
out.sample(1000).to_csv(OUT_DIR / "CLV_predictions_sample.csv", index=False)

print("\nSaved:")
print(OUT_DIR / f"clv_model_{best_name}.joblib")
print(OUT_DIR / "CLV_predictions_full.csv")
print(OUT_DIR / "CLV_predictions_sample.csv")

print("\nDone! 🚀  Your CLV model is trained & stored.")


Loading CLV dataset: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv
Dataset shape: (65911, 137)

Training Tweedie model...
Tweedie RMSE=23.3044 MAE=8.2479 R2=0.8794 Lift=5.058

Training log1p model...
Log1p RMSE=23.5068 MAE=8.1366 R2=0.8773 Lift=5.040

Selected Best Model: TWEEDIE
RMSE=23.3044 MAE=8.2479 R2=0.8794 LIFT=5.058

Saved:
C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\final_clv_model\clv_model_tweedie.joblib
C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\final_clv_model\CLV_predictions_full.csv
C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\final_clv_model\CLV_predictions_sample.csv

Done! 🚀  Your CLV model is trained & stored.


In [21]:
# Residual Booster: train small LGBM on residuals and add to original predictions
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------- CONFIG ----------------
CLV_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv"
TARGET = "future_30d_spend_true"
OUT_DIR = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params")
MODEL1_PATH = OUT_DIR / "lgb_best_expected_tweedie.joblib"   # first model you saved
OUT_DIR2 = OUT_DIR / "residual_booster"
OUT_DIR2.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.2

# ---------------- Load dataset & preprocess (same as before) ----------------
print("Loading dataset:", CLV_PATH)
df = pd.read_csv(CLV_PATH)
if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' not found in dataset columns: {df.columns.tolist()}")

y = df[TARGET].astype(float).values

# drop non-feature cols
drop_cols = ["person", "anchor_time", "time", "time_parsed", TARGET]
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore").copy()

# factorize non-numeric columns same as training
for c in X_df.select_dtypes(exclude=[np.number]).columns:
    X_df[c] = pd.factorize(X_df[c].astype(str))[0]

# impute median (re-fit here to match previous run; if you saved imputer use that instead)
IMP_PATH = OUT_DIR / "imputer_best_params.joblib"
if IMP_PATH.exists():
    imputer = joblib.load(IMP_PATH)
    X = pd.DataFrame(imputer.transform(X_df), columns=X_df.columns).astype("float32")
    print("Loaded imputer from:", IMP_PATH)
else:
    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X_df), columns=X_df.columns).astype("float32")
    joblib.dump(imputer, OUT_DIR2 / "imputer_residual.joblib")
    print("Saved new imputer to:", OUT_DIR2 / "imputer_residual.joblib")

# Train/test split (same random_state as before)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

# ---------------- Load first model ----------------
if MODEL1_PATH.exists():
    model1 = joblib.load(MODEL1_PATH)
    print("Loaded base model:", MODEL1_PATH)
else:
    raise FileNotFoundError(f"Base model not found at {MODEL1_PATH}. Please point MODEL1_PATH to your first model.")

# ---------------- Baseline predictions (model1) ----------------
pred1_train = model1.predict(X_train)
pred1_test  = model1.predict(X_test)

# Clip negatives (spend cannot be negative)
pred1_train = np.clip(pred1_train, 0.0, None)
pred1_test  = np.clip(pred1_test, 0.0, None)

# Baseline metrics on test (for comparison)
rmse_base = mean_squared_error(y_test, pred1_test) ** 0.5
mae_base  = mean_absolute_error(y_test, pred1_test)
r2_base   = r2_score(y_test, pred1_test)

def top_decile_lift(y_true, y_pred, decile=0.1):
    n = len(y_true)
    k = max(1, int(n * decile))
    idx = np.argsort(-y_pred)[:k]
    top_sum = y_true[idx].sum()
    baseline = y_true.mean() * k
    lift = top_sum / baseline if baseline > 0 else np.nan
    return {"lift": lift, "top_sum": top_sum, "baseline": baseline, "k": k}

lift_base = top_decile_lift(y_test, pred1_test, decile=0.1)

print("\nBASELINE (model1) test metrics:")
print(f"RMSE: {rmse_base:.4f}, MAE: {mae_base:.4f}, R2: {r2_base:.4f}, Top-decile lift: {lift_base['lift']:.3f}")

# ---------------- Train residual model ----------------
# residual = actual - pred1 (we model residual directly)
resid_train = y_train - pred1_train

# Option: clamp extreme residuals to stabilize training (optional)
# cap = np.percentile(np.abs(resid_train), 99)
# resid_train = np.clip(resid_train, -cap, cap)

# small, focused LGBM for residuals
residual_params = {
    "objective": "regression",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "lambda_l1": 0.1,
    "lambda_l2": 0.0,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1
}

print("\nTraining residual model (small LGBM)...")
model_resid = lgb.LGBMRegressor(**residual_params)
model_resid.fit(X_train, resid_train)  # no early stopping as requested

# ---------------- Predict residual corrections ----------------
pred_resid_test = model_resid.predict(X_test)
pred_resid_train = model_resid.predict(X_train)

# Compose final predictions
pred_final_test = pred1_test + pred_resid_test
pred_final_train = pred1_train + pred_resid_train

# Clip final to non-negative
pred_final_test = np.clip(pred_final_test, 0.0, None)
pred_final_train = np.clip(pred_final_train, 0.0, None)

# ---------------- Evaluate boosted model ----------------
rmse_final = mean_squared_error(y_test, pred_final_test) ** 0.5
mae_final  = mean_absolute_error(y_test, pred_final_test)
r2_final   = r2_score(y_test, pred_final_test)
lift_final = top_decile_lift(y_test, pred_final_test, decile=0.1)

print("\nBOOSTED (model1 + residual) test metrics:")
print(f"RMSE: {rmse_final:.4f}, MAE: {mae_final:.4f}, R2: {r2_final:.4f}, Top-decile lift: {lift_final['lift']:.3f}")

# ---------------- Save second model and final preds ----------------
MODEL2_PATH = OUT_DIR2 / "lgb_residual_model.joblib"
joblib.dump(model_resid, MODEL2_PATH)
print("Saved residual model to:", MODEL2_PATH)

OUT_PRED_CSV = OUT_DIR2 / "clv_preds_residual_boosted_full.csv"
out_df = df[["person"]].copy() if "person" in df.columns else pd.DataFrame({"row_id": np.arange(len(df))})
out_df = out_df.reset_index(drop=True)
# full preds use model1 + model2 on X (predict both)
pred1_full = model1.predict(X)
pred2_full = model_resid.predict(X)
pred_final_full = np.clip(pred1_full + pred2_full, 0.0, None)
out_df["pred_base_30d"] = np.clip(pred1_full, 0.0, None)
out_df["pred_resid_30d"] = pred2_full
out_df["pred_final_30d"] = pred_final_full
out_df["actual_30d"] = y
out_df.to_csv(OUT_PRED_CSV, index=False)
print("Saved full boosted predictions to:", OUT_PRED_CSV)

# ---------------- Quick feature importance of residual model ----------------
try:
    fi = pd.DataFrame({"feature": X.columns, "importance": model_resid.feature_importances_}).sort_values("importance", ascending=False)
    fi.to_csv(OUT_DIR2 / "residual_feature_importance.csv", index=False)
    print("Saved residual model feature importances to:", OUT_DIR2 / "residual_feature_importance.csv")
except Exception as e:
    print("Could not save residual feature importances:", e)

print("\nDone.")


Loading dataset: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv
Loaded imputer from: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\imputer_best_params.joblib
Train rows: 52728, Test rows: 13183
Loaded base model: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\lgb_best_expected_tweedie.joblib

BASELINE (model1) test metrics:
RMSE: 23.4560, MAE: 8.2592, R2: 0.8778, Top-decile lift: 5.043

Training residual model (small LGBM)...

BOOSTED (model1 + residual) test metrics:
RMSE: 23.4466, MAE: 8.2588, R2: 0.8779, Top-decile lift: 5.043
Saved residual model to: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\residual_booster\lgb_residual_model.joblib
Saved full boosted predictions to: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\residual_booster\clv_preds_residual_boosted_full

In [24]:
# fixed_catboost_ensemble_v3.py
# Robust end-to-end: factorize -> imputer -> cast categorical cols to string for CatBoost -> train -> ensemble

import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------- CONFIG ----------------
CLV_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv"
TARGET = "future_30d_spend_true"

LGB_MODEL_PATH = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\lgb_best_expected_tweedie.joblib")
SAVED_IMPUTER_PATH = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\imputer_best_params.joblib")

OUT_DIR = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\catboost_ensemble_v3")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPINGS_PATH = OUT_DIR / "cat_mappings_v3.joblib"

RANDOM_STATE = 42
TEST_SIZE = 0.2

# CatBoost quick hyperparams
cb_params = {
    "iterations": 3000,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 3,
    "loss_function": "RMSE",
    "random_seed": RANDOM_STATE,
    "verbose": 200,
    "thread_count": -1
}

# ---------------- Helpers ----------------
def rmse(a, b): return mean_squared_error(a, b) ** 0.5
def top_decile_lift(y_true, y_pred, decile=0.1):
    n = len(y_true)
    k = max(1, int(n * decile))
    idx = np.argsort(-y_pred)[:k]
    top_sum = y_true[idx].sum()
    baseline = y_true.mean() * k
    lift = top_sum / baseline if baseline > 0 else np.nan
    return {"lift": lift, "top_sum": top_sum, "baseline": baseline, "k": k}

# ---------------- Load data ----------------
print("Loading dataset:", CLV_PATH)
df = pd.read_csv(CLV_PATH)
if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' not found in dataset. Columns: {df.columns.tolist()}")

y = df[TARGET].astype(float).values
drop_cols = ["person", "anchor_time", "time", "time_parsed", TARGET]
X_df_raw = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore").copy()

# ---------------- Deterministic factorization & mapping ----------------
# Identify object/categorical columns
cat_cols = [c for c in X_df_raw.columns if X_df_raw[c].dtype == "object" or X_df_raw[c].dtype.name == "category"]

X_df_fact = X_df_raw.copy()
cat_mappings = {}
for c in cat_cols:
    codes, uniques = pd.factorize(X_df_fact[c].astype(str), sort=False)
    X_df_fact[c] = codes  # integer codes
    cat_mappings[c] = list(uniques)  # save unique values

joblib.dump(cat_mappings, MAPPINGS_PATH)
print("Saved cat mappings to:", MAPPINGS_PATH)
print("Categorical columns:", cat_cols)

# ---------------- Imputer: load or fit on factorized data ----------------
if SAVED_IMPUTER_PATH.exists():
    imputer = joblib.load(SAVED_IMPUTER_PATH)
    print("Loaded saved imputer:", SAVED_IMPUTER_PATH)
else:
    imputer = SimpleImputer(strategy="median")
    imputer.fit(X_df_fact)
    joblib.dump(imputer, OUT_DIR / "imputer_fitted_v3.joblib")
    print("Fitted & saved imputer to:", OUT_DIR / "imputer_fitted_v3.joblib")

# Transform factorized data to create X_lgb (numeric) and X_cb (mixed)
X_lgb = pd.DataFrame(imputer.transform(X_df_fact), columns=X_df_fact.columns).astype("float32")

# Build X_cb: keep categorical columns as strings, numeric columns filled by imputer
X_cb = X_df_fact.copy().astype(object)  # start as object so we can mix dtypes
# Fill numeric columns using imputer.statistics_
if hasattr(imputer, "statistics_"):
    stats = np.asarray(imputer.statistics_, dtype=float)
    for i, col in enumerate(X_df_fact.columns):
        col_vals = X_cb[col].values
        # replace NaN-like with imputer stat (but factorized data shouldn't have NaN; kept for safety)
        mask = pd.isna(col_vals)
        if mask.any():
            X_cb[col] = np.where(mask, stats[i], col_vals)
# Now cast categorical columns to strings (CatBoost accepts strings or ints)
for c in cat_cols:
    # ensure no NaN and force string representation
    X_cb[c] = X_cb[c].astype(int).astype(str)

# Now numeric columns should be numeric-like; make the numeric columns floats
num_cols = [c for c in X_cb.columns if c not in cat_cols]
X_cb[num_cols] = X_cb[num_cols].astype(float)

# CatBoost accepts cat feature names
cat_feature_names = [c for c in cat_cols if c in X_cb.columns]
print("Prepared X_lgb and X_cb. Cat features for CatBoost:", cat_feature_names)

# ---------------- Train/test split (consistent) ----------------
X_lgb_train, X_lgb_test, X_cb_train, X_cb_test, y_train, y_test = train_test_split(
    X_lgb, X_cb, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print("Train rows:", X_lgb_train.shape[0], "Test rows:", X_lgb_test.shape[0])

# ---------------- Train CatBoost on log1p target ----------------
y_train_log = np.log1p(np.clip(y_train, 0.0, None))
cb = CatBoostRegressor(**cb_params)
print("Training CatBoost (log1p target) ...")
# pass cat feature names
cb.fit(X_cb_train, y_train_log, cat_features=cat_feature_names)

# Predictions invert
pred_cb_test = np.expm1(cb.predict(X_cb_test))
pred_cb_full = np.expm1(cb.predict(X_cb))
pred_cb_test = np.clip(pred_cb_test, 0.0, None)
pred_cb_full = np.clip(pred_cb_full, 0.0, None)

# ---------------- Load LGB model and predict ----------------
if not LGB_MODEL_PATH.exists():
    raise FileNotFoundError(f"LGB model not found at {LGB_MODEL_PATH}")
lgb_model = joblib.load(LGB_MODEL_PATH)
pred_lgb_test = np.clip(lgb_model.predict(X_lgb_test), 0.0, None)
pred_lgb_full = np.clip(lgb_model.predict(X_lgb), 0.0, None)

# ---------------- Evaluate individual models ----------------
rmse_lgb = rmse(y_test, pred_lgb_test); r2_lgb = r2_score(y_test, pred_lgb_test); mae_lgb = mean_absolute_error(y_test, pred_lgb_test); lift_lgb = top_decile_lift(y_test, pred_lgb_test)
rmse_cb  = rmse(y_test, pred_cb_test);  r2_cb  = r2_score(y_test, pred_cb_test);  mae_cb  = mean_absolute_error(y_test, pred_cb_test);  lift_cb  = top_decile_lift(y_test, pred_cb_test)

print(f"\nLGB -> RMSE: {rmse_lgb:.4f}, MAE: {mae_lgb:.4f}, R2: {r2_lgb:.4f}, Lift: {lift_lgb['lift']:.3f}")
print(f"CB  -> RMSE: {rmse_cb:.4f}, MAE: {mae_cb:.4f}, R2: {r2_cb:.4f}, Lift: {lift_cb['lift']:.3f}")

# ---------------- Ensemble search ----------------
best_w = None; best_r2 = -9.0
for w in np.linspace(0, 1, 21):
    ens_test = w * pred_lgb_test + (1.0 - w) * pred_cb_test
    r2_e = r2_score(y_test, ens_test)
    if r2_e > best_r2:
        best_r2 = r2_e; best_w = float(w)

ens_test = best_w * pred_lgb_test + (1.0 - best_w) * pred_cb_test
rmse_ens = rmse(y_test, ens_test); mae_ens = mean_absolute_error(y_test, ens_test); r2_ens = r2_score(y_test, ens_test); lift_ens = top_decile_lift(y_test, ens_test)

print(f"\nBest ensemble weight w={best_w:.2f}")
print(f"Ensemble -> RMSE: {rmse_ens:.4f}, MAE: {mae_ens:.4f}, R2: {r2_ens:.4f}, Lift: {lift_ens['lift']:.3f}")

# ---------------- Save artifacts ----------------
joblib.dump(cb, OUT_DIR / "catboost_log1p_v3.joblib")
joblib.dump(imputer, OUT_DIR / "imputer_used_v3.joblib")
joblib.dump(lgb_model, OUT_DIR / "lgb_model_copied_v3.joblib")

ens_full = best_w * pred_lgb_full + (1.0 - best_w) * pred_cb_full
out_df = df[["person"]].copy() if "person" in df.columns else pd.DataFrame({"row_id": np.arange(len(df))})
out_df = out_df.reset_index(drop=True)
out_df["pred_lgb"] = pred_lgb_full
out_df["pred_catboost"] = pred_cb_full
out_df["pred_ensemble"] = ens_full
out_df["actual_30d"] = y
out_df.to_csv(OUT_DIR / "clv_preds_catboost_ensemble_v3_full.csv", index=False)

print("Saved CatBoost model and ensemble preds to:", OUT_DIR)
print("Done.")


Loading dataset: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv
Saved cat mappings to: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\catboost_ensemble_v3\cat_mappings_v3.joblib
Categorical columns: ['gender', 'became_member_on', 'behavior_segment', 'loyalty_tier', 'recency_bucket', 'anchor_time_parsed']
Loaded saved imputer: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\imputer_best_params.joblib
Prepared X_lgb and X_cb. Cat features for CatBoost: ['gender', 'became_member_on', 'behavior_segment', 'loyalty_tier', 'recency_bucket', 'anchor_time_parsed']
Train rows: 52728 Test rows: 13183
Training CatBoost (log1p target) ...
0:	learn: 2.0791993	total: 228ms	remaining: 11m 23s
200:	learn: 0.1665109	total: 23.4s	remaining: 5m 25s
400:	learn: 0.1628358	total: 44.3s	remaining: 4m 46s
600:	learn: 0.1587388	total: 1m 6s	remaining: 4m 23s
800:	learn: 0.1551505	total: 1m 27s	remaini

In [26]:
# stacking_lgb_catboost_ridge.py
# Build OOF preds for LGB (Tweedie) and CatBoost (Tweedie), then train Ridge meta-learner.
# Manual metrics computed. Uses 3-fold OOF by default (fast).

import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import time

from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb
from catboost import CatBoostRegressor

# ---------------- CONFIG ----------------
CLV_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv"
TARGET = "future_30d_spend_true"
OUT_DIR = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\stacking_ridge")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 3           # 3 is faster; increase to 5 for better OOF estimates
LGB_ROUNDS = 1500     # keep reasonable so fold time is manageable
CB_ITERS = 1500       # CatBoost iterations for fold fits
USE_TWEEDIE = True    # use Tweedie objective for both bases

# load saved preprocessing artifacts if available (safe to continue without)
SAVED_IMPUTER = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\best_expected_params\imputer_best_params.joblib")
MAPPINGS_PATH = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\catboost_tweedie_hpt\cat_mappings.joblib")

# ---------------- helpers ----------------
def rmse(a,b): return mean_squared_error(a,b)**0.5
def top_decile_lift(y_true, y_pred, decile=0.1):
    n = len(y_true)
    k = max(1, int(n*decile))
    idx = np.argsort(-y_pred)[:k]
    top_sum = y_true[idx].sum()
    baseline = y_true.mean() * k
    return top_sum / baseline if baseline>0 else np.nan

# ---------------- Load data ----------------
print("Loading dataset:", CLV_PATH)
df = pd.read_csv(CLV_PATH)
if TARGET not in df.columns:
    raise ValueError("Target not found.")

y = df[TARGET].astype(float).values
drop_cols = ["person", "anchor_time", "time", "time_parsed", TARGET]
X_df_raw = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore").copy()

# load/save factorize mappings (we'll factorize deterministically)
cat_cols = [c for c in X_df_raw.columns if X_df_raw[c].dtype == "object" or X_df_raw[c].dtype.name == "category"]
cat_mappings = {}
X_df_fact = X_df_raw.copy()
for c in cat_cols:
    codes, uniques = pd.factorize(X_df_fact[c].astype(str), sort=False)
    X_df_fact[c] = codes
    cat_mappings[c] = list(uniques)
joblib.dump(cat_mappings, OUT_DIR / "cat_mappings_oof.joblib")

# ---------------- Impute numeric features (works on factorized data) ----------------
if SAVED_IMPUTER.exists():
    imputer = joblib.load(SAVED_IMPUTER)
else:
    imputer = SimpleImputer(strategy="median")
    imputer.fit(X_df_fact)
    joblib.dump(imputer, OUT_DIR / "imputer_oof_fitted.joblib")

X_numeric = pd.DataFrame(imputer.transform(X_df_fact), columns=X_df_fact.columns)

# For CatBoost training we prefer categorical columns as strings
X_cb = X_df_fact.copy().astype(object)
for c in cat_cols:
    X_cb[c] = X_cb[c].astype(int).astype(str)
# fill numeric columns if any NaNs
num_cols = [c for c in X_cb.columns if c not in cat_cols]
X_cb[num_cols] = X_cb[num_cols].astype(float)
# Cat feature names list
cat_feature_names = list(cat_cols)

# ---------------- Prepare OOF arrays ----------------
n = len(df)
oof_pred_lgb = np.zeros(n, dtype=float)
oof_pred_cb  = np.zeros(n, dtype=float)
test_idx_all = np.arange(n)  # for final full-model training/prediction

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fold_no = 0
start_time = time.time()
for train_idx, valid_idx in kf.split(X_numeric):
    fold_no += 1
    print(f"\n--- Fold {fold_no}/{N_FOLDS} ({len(train_idx)} train / {len(valid_idx)} val) ---")
    Xtr_num, Xval_num = X_numeric.iloc[train_idx], X_numeric.iloc[valid_idx]
    Xtr_cb,  Xval_cb  = X_cb.iloc[train_idx], X_cb.iloc[valid_idx]
    ytr, yval = y[train_idx], y[valid_idx]

    # ---------- LightGBM base (Tweedie or reg) ----------
    lgb_params = {
        "objective": "tweedie" if USE_TWEEDIE else "regression",
        "tweedie_variance_power": 1.4 if USE_TWEEDIE else None,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_data_in_leaf": 20,
        "lambda_l1": 0.5,
        "lambda_l2": 0.0,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbosity": -1,
        "n_jobs": -1
    }
    lgbm = lgb.LGBMRegressor(n_estimators=LGB_ROUNDS, **{k:v for k,v in lgb_params.items() if v is not None})
    lgbm.fit(Xtr_num, ytr)
    pred_val_lgb = lgbm.predict(Xval_num)
    pred_val_lgb = np.clip(pred_val_lgb, 0.0, None)
    oof_pred_lgb[valid_idx] = pred_val_lgb

    # ---------- CatBoost base (Tweedie) ----------
    cb_params = {
        "iterations": CB_ITERS,
        "learning_rate": 0.03,
        "depth": 6,
        "loss_function": "Tweedie:variance_power=1.4" if USE_TWEEDIE else "RMSE",
        "random_seed": RANDOM_STATE,
        "thread_count": -1,
        "verbose": False
    }
    cb = CatBoostRegressor(**cb_params)
    # Train on raw y when using Tweedie loss
    cb.fit(Xtr_cb, ytr, cat_features=cat_feature_names)
    pred_val_cb = cb.predict(Xval_cb)
    pred_val_cb = np.clip(pred_val_cb, 0.0, None)
    oof_pred_cb[valid_idx] = pred_val_cb

    # per-fold quick metrics
    print("Fold metrics LGB: RMSE={:.4f}, R2={:.4f}".format(rmse(yval, pred_val_lgb), r2_score(yval, pred_val_lgb)))
    print("Fold metrics CB : RMSE={:.4f}, R2={:.4f}".format(rmse(yval, pred_val_cb), r2_score(yval, pred_val_cb)))

elapsed = time.time() - start_time
print("\nOOF generation done in {:.1f}s".format(elapsed))

# ---------------- Meta-features and meta-learner training ----------------
meta_X = np.vstack([oof_pred_lgb, oof_pred_cb, oof_pred_cb - oof_pred_lgb]).T
meta_y = y.copy()
# use RidgeCV with alphas grid (fast)
alphas = [0.1, 1.0, 10.0, 50.0]
meta_model = RidgeCV(alphas=alphas, store_cv_values=False)
meta_model.fit(meta_X, meta_y)
print("Trained Ridge meta-learner. Coefs:", meta_model.coef_, "alpha:", meta_model.alpha_)

# ---------------- Evaluate OOF stacked predictions ----------------
oof_stack = meta_model.predict(meta_X)
rmse_oof = rmse(y, oof_stack)
mae_oof = mean_absolute_error(y, oof_stack)
r2_oof = r2_score(y, oof_stack)
lift_oof = top_decile_lift(y, oof_stack)
print("\nOOF STACK metrics: RMSE={:.4f}, MAE={:.4f}, R2={:.4f}, Lift={:.3f}".format(rmse_oof, mae_oof, r2_oof, lift_oof))

# ---------------- Retrain base learners on full data and predict final ensemble on full set ----------------
print("\nRetraining base learners on full data for final preds...")
# LGB on full numeric X
final_lgb = lgb.LGBMRegressor(n_estimators=LGB_ROUNDS, **{k:v for k,v in lgb_params.items() if v is not None})
final_lgb.fit(X_numeric, y)
pred_lgb_full = np.clip(final_lgb.predict(X_numeric), 0.0, None)

# CatBoost on full X_cb
final_cb = CatBoostRegressor(iterations=CB_ITERS, learning_rate=0.03, depth=6,
                             loss_function="Tweedie:variance_power=1.4" if USE_TWEEDIE else "RMSE",
                             random_seed=RANDOM_STATE, thread_count=-1, verbose=False)
final_cb.fit(X_cb, y, cat_features=cat_feature_names)
pred_cb_full = np.clip(final_cb.predict(X_cb), 0.0, None)

# Compose final meta features and predict
meta_full = np.vstack([pred_lgb_full, pred_cb_full, pred_cb_full - pred_lgb_full]).T
pred_stack_full = meta_model.predict(meta_full)
pred_stack_full = np.clip(pred_stack_full, 0.0, None)

# Evaluate on full set (you can also slice a held-out test instead; here full metrics are shown)
rmse_full = rmse(y, pred_stack_full)
mae_full = mean_absolute_error(y, pred_stack_full)
r2_full = r2_score(y, pred_stack_full)
lift_full = top_decile_lift(y, pred_stack_full)

print("\nFINAL STACK (full) metrics: RMSE={:.4f}, MAE={:.4f}, R2={:.4f}, Lift={:.3f}".format(rmse_full, mae_full, r2_full, lift_full))

# ---------------- Save models & preds ----------------
joblib.dump(final_lgb, OUT_DIR / "final_lgb_for_stack.joblib")
joblib.dump(final_cb, OUT_DIR / "final_cb_for_stack.joblib")
joblib.dump(meta_model, OUT_DIR / "ridge_meta.joblib")

out_df = pd.DataFrame({"person": df["person"] if "person" in df.columns else np.arange(n),
                       "pred_lgb": pred_lgb_full, "pred_cb": pred_cb_full, "pred_stack": pred_stack_full, "actual": y})
out_df.to_csv(OUT_DIR / "stack_preds_full.csv", index=False)
print("Saved models and preds to:", OUT_DIR)


Loading dataset: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv

--- Fold 1/3 (43940 train / 21971 val) ---
Fold metrics LGB: RMSE=23.4679, R2=0.8780
Fold metrics CB : RMSE=23.0384, R2=0.8824

--- Fold 2/3 (43941 train / 21970 val) ---
Fold metrics LGB: RMSE=23.2665, R2=0.8790
Fold metrics CB : RMSE=22.7934, R2=0.8839

--- Fold 3/3 (43941 train / 21970 val) ---
Fold metrics LGB: RMSE=22.8871, R2=0.8815
Fold metrics CB : RMSE=22.4575, R2=0.8859

OOF generation done in 498.1s
Trained Ridge meta-learner. Coefs: [0.46888244 0.55913096 0.09024852] alpha: 50.0

OOF STACK metrics: RMSE=22.6650, MAE=7.9131, R2=0.8851, Lift=5.113

Retraining base learners on full data for final preds...

FINAL STACK (full) metrics: RMSE=11.6398, MAE=4.3169, R2=0.9697, Lift=5.405
Saved models and preds to: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\CLV\stacking_ridge
